# Chunk 52 — Replay-based fine-tuning of the GATv2 on the balanced single-band datasets (Chunk 51)

**Why:** Chunk 51 built new 5,000-antenna datasets for 12×12, 15×15, 35×35 and 45×45: about 50% single-band and 50% non-functioning, split 4,000 / 500 / 500. About 80% of the new val/test antennas were in the training pools of every earlier 12/15/35/45 model. This chunk therefore **restarts from `best_model.pt`**, which saw only 25×25, and repeats the original training order on the new data.

**Chain (recipes from Chunks 30/42/44/50, unchanged):**

| stage | starts from | trains on (per epoch) | replay per epoch | selection |
|---|---|---|---|---|
| A `bal_ft12` | `best_model.pt`, widened to 6 inputs (zero column, exact) | new 12×12 train (4,000) | 4,000 each from 15 / 25 / 35 / 45 | new 12 val MSE |
| B `bal_ft15` | stage A | new 15×15 train (4,000) | 4,000 each from 12 / 25 / 35 / 45 | new 15 val MSE |
| C `bal_ft35` | stage B | new 35×35 train (4,000) | 4,000 each from 12 / 15 / 25 / 45 | new 35 val MSE |
| D `bal_ft45` | stage C | new 45×45 train (4,000) | 4,000 each from 12 / 15 / 25 / 35 | new 45 val MSE |

Every stage sees all five grids each epoch: its target grid in full, plus an equal-sized replay draw from each of the other four. Stage D's model is the **final five-grid model**.

- **Optimiser:** Adam, lr 1e-4, weight decay 1e-4, batch 64, gradient clipping 1.0.
- **Schedule:** ReduceLROnPlateau (factor 0.5, patience 5); early stopping with patience 10; at most 60 epochs.
- **Seeds:** torch 1000+s; replay sampling 5200 / 5300 / 5400 / 5500 + s for stages A / B / C / D. The default is seed 0.
- **Replay sources:** 12/15/35/45 replay comes from the **new** Chunk 51 train sets only. 25×25 replay comes from its unchanged `indices.json` train split; 25×25 is not touched.
- **Normalisation:** unchanged (the 25×25 `s11_mean` / `s11_std`).

**Test (one touch):**
- **Every stage** on the new 12/15/35/45 test sets (500 each) and the unchanged 25×25 test set.
- **`best_model.pt` zero-shot** on the same sets, as the reference. On 25×25 it must reproduce Chunk 36's recorded test MSE 1.3175 and MAE 0.3742.
- **Errors split by class** (single-band vs non-functioning), plus AUROC.
- **Full predicted curves** saved for every test antenna, so the Gupta CNN chunk and later figures can do paired comparisons without these checkpoints.

**Pre-registered statements:**

| | question |
|---|---|
| S1 | Does the final model beat zero-shot `best_model` on each new test set (paired MSE and MAE)? |
| S2 | How much 25×25 accuracy is lost through the chain (final vs `best_model`, paired)? |
| S3 | On each grid, how do single-band and non-functioning errors compare (final model)? |

**Results apply to single-band and non-functioning antennas only, and are not comparable with the natural-distribution table.**

**Writes:**
- `ch52_*` artifacts: runs, curves, per-sample with predicted curves, metrics, paired comparisons, report
- `checkpoints/ch52_*.pt`

## 1. Installs

In [1]:
!pip -q install torch_geometric pyarrow tabulate

## 2. Repository and Drive

In [2]:
import os, sys, subprocess
REPO_ROOT = '/content/antenna_gnn'
if os.path.isdir(f'{REPO_ROOT}/.git'):
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/asparagusD/antenna_gnn.git', REPO_ROOT], check=True)
sys.path.insert(0, f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
ART, CKPT, SPLITS, LOCAL = f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints', f'{DATA_ROOT}/splits', '/content/local_graphs'
os.makedirs(LOCAL, exist_ok=True)
print(REPO_ROOT, DATA_ROOT)

Mounted at /content/drive
/content/antenna_gnn /content/drive/MyDrive/antenna_gnn


## 3. Setup
- **Determinism:** deterministic cuDNN, TF32 off.
- **Provenance:** hashes every input file and captures the mtimes of earlier files, so the guards can prove nothing earlier was changed.
- **Statistics:** loads the canonical 25×25 normalisation statistics read-only.

In [3]:
# ── Setup: scope, provenance, determinism ──
import json, glob, hashlib, zipfile, shutil, time, copy
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from model import AntennaGNN
from feed_component import AppendFeedComponentFeature

TAU, F = -10.0, np.linspace(1, 4, 201)
NEW = (12, 15, 35, 45); GRIDS = (12, 15, 25, 35, 45)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); assert DEVICE.type == 'cuda', 'GPU required'
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
READ_SHA = {}
def READ(p):
    assert os.path.exists(p), f'missing {p}'
    READ_SHA.setdefault(p, sha(p)); return p
def OUT(n):
    assert n.startswith('ch52_'), n; return f'{ART}/{n}'
def COUT(n):
    assert n.startswith('ch52_') and n.endswith('.pt'), n; return f'{CKPT}/{n}'
PRIOR = {p: os.path.getmtime(p) for pat in (f'{ART}/*', f'{SPLITS}/*', f'{CKPT}/*.pt') for p in glob.glob(pat)
         if os.path.isfile(p) and not Path(p).name.startswith('ch52_')}
MEAN = torch.tensor(np.load(READ(f'{ART}/s11_mean.npy')), dtype=torch.float32)   # canonical 25x25 statistics, read only
STD = torch.tensor(np.load(READ(f'{ART}/s11_std.npy')), dtype=torch.float32)
MEAN_D, STD_D = MEAN.to(DEVICE), STD.to(DEVICE)
BASE_PATH = f'{CKPT}/best_model.pt'
TEST_TOUCHES = 0
print(f'GPU: {torch.cuda.get_device_name(0)} | captured mtimes of {len(PRIOR)} earlier files')

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | captured mtimes of 298 earlier files


## 4. Splits
**Sources:**
- **New 12/15/35/45:** the Chunk 51 split files. Each entry is `[source, sample_idx]`, and `35x35_extra` is Chunk 45's zip.
- **25×25:** its unchanged `indices.json` split.

**Checks:** the new sets must be exactly 4,000 / 500 / 500 with no overlap, and the class labels must match `ch51_manifest.parquet`.

In [4]:
# ── Splits: Chunk 51 for 12/15/35/45, unchanged indices.json for 25x25 ──
S = {}
for N in NEW:
    S[N] = {r: [(str(a), int(b)) for a, b in json.load(open(READ(f'{SPLITS}/ch51_{N}x{N}_{r}.json')))] for r in ('train', 'val', 'test')}
    a, b, c = (set(S[N][r]) for r in ('train', 'val', 'test'))
    assert (len(a), len(b), len(c)) == (4000, 500, 500) and not (a & b or a & c or b & c), f'STOP: {N}x{N} Chunk 51 splits'
IDX = json.load(open(READ(f'{SPLITS}/indices.json')))
def pairs(v):
    if isinstance(v, dict):
        return {(int(g), int(i)) for g, x in v.items() for i in x}
    return {(int(e[0]), int(e[1])) for e in v}
S[25] = {r: [('25x25', i) for i in sorted(i for g, i in pairs(IDX[k]) if g == 25)] for r, k in (('train', 'train'), ('val', 'val'), ('test', 'test'))}
assert tuple(len(S[25][r]) for r in ('train', 'val', 'test')) == (79866, 9984, 9983), 'STOP: 25x25 splits changed'
MAN51 = pd.read_parquet(READ(f'{ART}/ch51_manifest.parquet'))
CLS = {(r.source, int(r.sample_idx)): r.cls for r in MAN51[MAN51.selected].itertuples()}
for N in NEW:
    for r in ('train', 'val', 'test'):
        assert all(k in CLS for k in S[N][r]), f'STOP: {N}x{N} {r} has antennas not selected in Chunk 51'
for N in GRIDS:
    print(f'{N}x{N}: train {len(S[N]["train"]):,} | val {len(S[N]["val"]):,} | test {len(S[N]["test"]):,}'
          + (f' | test single-band {sum(CLS[k] == "single_band" for k in S[N]["test"])}' if N in NEW else ' (unchanged)'))

12x12: train 4,000 | val 500 | test 500 | test single-band 250
15x15: train 4,000 | val 500 | test 500 | test single-band 250
25x25: train 79,866 | val 9,984 | test 9,983 (unchanged)
35x35: train 4,000 | val 500 | test 500 | test single-band 254
45x45: train 4,000 | val 500 | test 500 | test single-band 260


## 5. Stage graphs and cache
**What it does:**
- Unzips the processed graphs for every grid, plus `processed_35x35_extra.zip`, to local SSD.
- Caches each graph in RAM the first time it's used, with the 8-connected feed-component column appended, as in Chunks 42–50. The base model's 5-input copy is kept separately for the zero-shot reference.

In [5]:
# ── Stage graphs; cache ──
SRC = [f'{N}x{N}' for N in GRIDS] + ['35x35_extra']
FOLDER = {s: f'{LOCAL}/{s}' for s in SRC}
ZIPS = {s: f'{DATA_ROOT}/data/processed/processed_{s}.zip' for s in SRC}
for s in SRC:
    d = FOLDER[s]
    if os.path.exists(f'{d}/_STAGED.txt'):
        continue
    os.makedirs(d, exist_ok=True)
    with zipfile.ZipFile(READ(ZIPS[s])) as Z:
        mem = [m for m in Z.namelist() if m.endswith('.pt')]
        for m in tqdm(mem, desc=f'stage {s}'):
            with Z.open(m) as a, open(f'{d}/{os.path.basename(m)}', 'wb') as b:
                shutil.copyfileobj(a, b)
    open(f'{d}/_STAGED.txt', 'w').write(str(len(mem)))
TRANSFORM = AppendFeedComponentFeature()
CACHE = {}
def graph(key, nf=6):
    # key = (source, sample_idx); returns a 6-feature graph (or the stored 5-feature one for the zero-shot reference)
    src, i = key
    if key not in CACHE:
        n_ = int(src.split('x')[0])
        d = torch.load(f'{FOLDER[src]}/sample_{int(i)}.pt', weights_only=False)
        assert d.x.shape == (n_ * n_ + 1, 5) and float(d.x[-1, 3]) == -1.0 and d.y.numel() == 201 and float(d.y.max()) <= 0.01
        g = TRANSFORM(Data(x=d.x.clone(), edge_index=d.edge_index, edge_attr=d.edge_attr))
        CACHE[key] = (g.x, g.edge_index, g.edge_attr, d.y.reshape(1, 201).float())
    x, ei, ea, y = CACHE[key]
    return Data(x=x if nf == 6 else x[:, :5], edge_index=ei, edge_attr=ea, y=(y - MEAN) / (STD + 1e-8), y_raw=y)
def truth(keys):
    return np.stack([graph(k).y_raw.reshape(-1).numpy() for k in keys]).astype(np.float64)
print('staged:', {s: open(f'{FOLDER[s]}/_STAGED.txt').read() for s in SRC})

stage 12x12:   0%|          | 0/7000 [00:00<?, ?it/s]

stage 15x15:   0%|          | 0/6367 [00:00<?, ?it/s]

stage 25x25:   0%|          | 0/99833 [00:00<?, ?it/s]

stage 35x35:   0%|          | 0/4988 [00:00<?, ?it/s]

stage 45x45:   0%|          | 0/6984 [00:00<?, ?it/s]

stage 35x35_extra:   0%|          | 0/1997 [00:00<?, ?it/s]

staged: {'12x12': '7000', '15x15': '6367', '25x25': '99833', '35x35': '4988', '45x45': '6984', '35x35_extra': '1997'}


## 6. Models: base, widened base, exactness guard
**What it does:**
- Loads `best_model.pt` (5 inputs, 587,001 parameters).
- Copies every weight into a 6-input model (587,129 parameters) with column 5 of `input_proj.weight` set to **zero**, exactly as Chunk 30 did.

**Exactness guard:** 30 graphs per grid, compared in **float64 on CPU**, must agree to < 1e-5 dB. A CPU comparison is used because GPU GATv2 is non-deterministic at about 1e-4 dB.

In [6]:
# ── Base model, zero-column widening, exactness guard ──
def gnn(nf):
    return AntennaGNN(node_feat_dim=nf, edge_feat_dim=2, hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201)
def n_par(m): return sum(p.numel() for p in m.parameters())
def state_of(path):
    c = torch.load(READ(path), map_location='cpu', weights_only=False)
    s = c['model_state'] if isinstance(c, dict) and 'model_state' in c else c
    return {k[7:] if k.startswith('module.') else k: v for k, v in s.items()}
BASE5 = gnn(5); BASE5.load_state_dict(state_of(BASE_PATH), strict=True); assert n_par(BASE5) == 587_001
lin5 = [n for n, m in BASE5.named_modules() if isinstance(m, nn.Linear) and m.in_features == 5]
assert lin5 == ['input_proj'], f'STOP: unexpected 5-input layers {lin5}'
BASE6 = gnn(6); sd = {k: v.clone() for k, v in BASE5.state_dict().items()}
w = torch.zeros(sd['input_proj.weight'].shape[0], 6); w[:, :5] = sd['input_proj.weight']; sd['input_proj.weight'] = w
BASE6.load_state_dict(sd, strict=True); assert n_par(BASE6) == 587_129
@torch.no_grad()
def cpu64(m, keys, nf):
    m = copy.deepcopy(m).double().cpu().eval(); b = next(iter(GeoLoader([graph(k, nf) for k in keys], batch_size=len(keys))))
    b.x, b.edge_attr = b.x.double(), b.edge_attr.double()
    return (m(b) * STD.double() + MEAN.double()).numpy()
worst = max(np.abs(cpu64(BASE6, S[N]['val'][:30], 6) - cpu64(BASE5, S[N]['val'][:30], 5)).max() for N in GRIDS)
print(f'zero-column widening: max |difference| over 150 graphs = {worst:.2e} dB')
assert worst < 1e-5, 'STOP: the widened model is not identical to best_model'
@torch.no_grad()
def predict(m, keys, nf=6, bs=128):
    m = m.to(DEVICE).eval(); out = []
    for s in range(0, len(keys), bs):
        gs = [graph(k, nf) for k in keys[s:s + bs]]
        out.append((m(next(iter(GeoLoader(gs, batch_size=len(gs)))).to(DEVICE)) * STD_D + MEAN_D).double().cpu())
    return torch.cat(out).numpy()

zero-column widening: max |difference| over 150 graphs = 2.84e-14 dB


## 7. VAL harness and zero-shot reference (before training)
Prints the zero-shot `best_model` val MSE on every new val set and on a fixed 2,000-antenna 25×25 val subset. These are the starting points the chain must improve on.

In [7]:
# ── VAL harness: zero-shot best_model on the new val sets ──
VAL25 = sorted(np.random.default_rng(42).choice(len(S[25]['val']), 2000, replace=False).tolist())
VAL = {N: S[N]['val'] for N in NEW}; VAL[25] = [S[25]['val'][j] for j in VAL25]
def vmse(m, N, nf=6):
    return float(((predict(m, VAL[N], nf) - truth(VAL[N])) ** 2).mean())
ZS_VAL = {N: vmse(BASE5, N, 5) for N in GRIDS}
for N in GRIDS:
    print(f'best_model zero-shot {N}x{N} VAL MSE {ZS_VAL[N]:.4f}' + ('  (2,000-antenna subset; in-distribution)' if N == 25 else ''))
HARNESS = True

best_model zero-shot 12x12 VAL MSE 5.3832
best_model zero-shot 15x15 VAL MSE 4.4591
best_model zero-shot 25x25 VAL MSE 1.3008  (2,000-antenna subset; in-distribution)
best_model zero-shot 35x35 VAL MSE 2.3074
best_model zero-shot 45x45 VAL MSE 2.9135


## 8. Train the chain (resumable)
**Queue:** stages A → B → C → D for each seed in `SEEDS`. The default is seed 0; set `SEEDS = (0, 1)` for a second seed.
- Each stage starts from the previous stage's checkpoint of the same seed; stage A starts from the widened `best_model`.
- A finished run is skipped; an interrupted run restarts from scratch.

**Every epoch:**
- **Data:** the stage's target train sets, plus fresh random replay draws.
- **Selection metric:** the stage's target val MSE, computed and used for the LR schedule and early stopping.
- **Forgetting monitor:** val MSE on every other grid, logged at epoch 1 and every 5 epochs.

**Checkpoints:** the best model is mirrored to Drive with a forced flush and verified by SHA-256.

**Writes:** `ch52_curves.csv`, `ch52_runs.csv`, `checkpoints/ch52_<run>.pt`.

In [9]:
# ── Training: A (12, replay 15/25/35/45) -> B (15, replay 12/25/35/45) -> C (35, replay 12/15/25/45) -> D (45, replay 12/15/25/35) ──
RECIPE = dict(lr=1e-4, wd=1e-4, batch=64, patience=10, max_ep=60)          # Chunk 42/44/50 replay recipe, unchanged
STAGES = {'bal_ft12': dict(targets=(12,), replay=(15, 25, 35, 45), per_grid=4000, rng=5200, init='base'),
          'bal_ft15': dict(targets=(15,), replay=(12, 25, 35, 45), per_grid=4000, rng=5300, init='bal_ft12'),
          'bal_ft35': dict(targets=(35,), replay=(12, 15, 25, 45), per_grid=4000, rng=5400, init='bal_ft15'),
          'bal_ft45': dict(targets=(45,), replay=(12, 15, 25, 35), per_grid=4000, rng=5500, init='bal_ft35')}
SEEDS = (0, 1)
QUEUE = [(arm, s) for s in SEEDS for arm in STAGES]
CURVES, RUNS = OUT('ch52_curves.csv'), OUT('ch52_runs.csv')
SESSION_HOURS = 10; T0S = time.perf_counter(); TIMES = []
def rid(arm, s): return f'{arm}_s{s}'
def ckpt(arm, s): return COUT(f'ch52_{rid(arm, s)}.pt')
def done(): return set(pd.read_csv(RUNS).run_id) if os.path.exists(RUNS) else set()
def append(df, p):
    # align by column NAME (stages log different val columns); a positional CSV append would misalign them
    if os.path.exists(p):
        df = pd.concat([pd.read_csv(p), df], ignore_index=True)
    df.to_csv(p, index=False)
def mirror(local, drive_path):
    shutil.copy2(local, drive_path); h = sha(local)
    drive.flush_and_unmount(); drive.mount('/content/drive', force_remount=True)
    assert sha(drive_path) == h, f'STOP: {Path(drive_path).name} did not persist'
    return h

def train(arm, s):
    r, out, st_ = rid(arm, s), ckpt(arm, s), STAGES[arm]
    if r in done() and os.path.exists(out):
        print('skip', r); return
    if st_['init'] == 'base':
        m, init_name = copy.deepcopy(BASE6), Path(BASE_PATH).name + ' (+zero column)'
    else:
        p = ckpt(st_['init'], s); assert os.path.exists(p), f'STOP: {r} needs {Path(p).name}; run the earlier stage first'
        m = gnn(6); m.load_state_dict(state_of(p), strict=True); init_name = Path(p).name
    torch.manual_seed(1000 + s); torch.cuda.manual_seed_all(1000 + s); rng = np.random.default_rng(st_['rng'] + s)
    m = m.to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=RECIPE['lr'], weight_decay=RECIPE['wd'])
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    best, bad, best_row, steps, t0 = float('inf'), 0, None, 0, time.perf_counter()
    best_local = f'/content/{r}.best'
    if os.path.exists(CURVES):
        C = pd.read_csv(CURVES); C[C.run_id != r].to_csv(CURVES, index=False)

    epoch_bar = tqdm(range(1, RECIPE['max_ep'] + 1), desc=f'Training {r}')
    for ep in epoch_bar:
        m.train(); tot, n = 0.0, 0
        keys = [k for N in st_['targets'] for k in S[N]['train']]
        for G in st_['replay']:
            pool = S[G]['train']; take = rng.choice(len(pool), st_['per_grid'], replace=len(pool) < st_['per_grid'])
            keys += [pool[j] for j in take]
        order = rng.permutation(len(keys))
        for b in GeoLoader([graph(keys[j]) for j in order], batch_size=RECIPE['batch'], shuffle=False):
            b = b.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss = nn.functional.mse_loss(m(b), b.y.view(-1, 201))
            loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            tot += loss.item() * b.num_graphs; n += b.num_graphs; steps += 1

        tv = {N: vmse(m, N) for N in st_['targets']}
        # Also calculate MAE for display metrics
        tv_mae = {N: float(np.abs(predict(m, VAL[N]) - truth(VAL[N])).mean()) for N in st_['targets']}

        sel = float(np.mean(list(tv.values()))); sch.step(sel)
        row = dict(run_id=r, arm=arm, seed=s, epoch=ep, steps=steps, wall_seconds=time.perf_counter() - t0, train_loss_norm=tot / n,
                   select_val_mse_db2=sel, lr=opt.param_groups[0]['lr'], **{f'val{N}_mse_db2': v for N, v in tv.items()})
        if ep == 1 or ep % 5 == 0:
            for G in GRIDS:
                if G not in st_['targets']:
                    row[f'val{G}_mse_db2'] = vmse(m, G)
        append(pd.DataFrame([row]), CURVES)
        if sel < best:
            best, bad, best_row = sel, 0, row
            torch.save({'model_state': m.state_dict(), 'arm': arm, 'seed': s, 'best_epoch': ep, 'best_select_val_mse': sel, 'init': init_name}, best_local)
        else:
            bad += 1

        # Update tqdm status bar
        metrics_str = ' '.join([f'val{N}_MSE: {tv[N]:.4f} val{N}_MAE: {tv_mae[N]:.4f}' for N in st_['targets']])
        epoch_bar.set_postfix_str(f'{metrics_str} | best_MSE: {best:.4f}')

        if bad >= RECIPE['patience']:
            break
    h = mirror(best_local, out)
    rec = dict(run_id=r, arm=arm, seed=s, targets=str(st_['targets']), init=init_name, replay=str(st_['replay']), replay_per_grid=st_['per_grid'],
               best_epoch=best_row['epoch'], epochs_run=ep, minutes=(time.perf_counter() - t0) / 60, select_val_mse_db2=best_row['select_val_mse_db2'],
               budget_limited=bool(best_row['epoch'] > RECIPE['max_ep'] - RECIPE['patience']), checkpoint=out, checkpoint_sha256=h,
               **{k: v for k, v in best_row.items() if k.startswith('val') and k.endswith('_mse_db2')})
    append(pd.DataFrame([rec]), RUNS); TIMES.append(rec['minutes'] * 60); os.remove(best_local)
    del m, opt; torch.cuda.empty_cache()
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rec.items() if k not in ('checkpoint', 'checkpoint_sha256')})

assert globals().get('HARNESS') is True
for job in QUEUE:
    used = (time.perf_counter() - T0S) / 3600
    if TIMES and used + max(TIMES) / 3600 > SESSION_HOURS:
        print(f'session budget reached ({used:.1f} h): re-run this cell in a new session to continue'); break
    train(*job)
print('completed:', len(done()), 'of', len(QUEUE))

skip bal_ft12_s0
skip bal_ft15_s0
skip bal_ft35_s0
skip bal_ft45_s0


Training bal_ft12_s1:   0%|          | 0/60 [00:00<?, ?it/s]

Mounted at /content/drive
{'run_id': 'bal_ft12_s1', 'arm': 'bal_ft12', 'seed': 1, 'targets': '(12,)', 'init': 'best_model.pt (+zero column)', 'replay': '(15, 25, 35, 45)', 'replay_per_grid': 4000, 'best_epoch': 36, 'epochs_run': 46, 'minutes': 16.7823, 'select_val_mse_db2': 1.8437, 'budget_limited': False, 'val12_mse_db2': 1.8437}


Training bal_ft15_s1:   0%|          | 0/60 [00:00<?, ?it/s]

Mounted at /content/drive
{'run_id': 'bal_ft15_s1', 'arm': 'bal_ft15', 'seed': 1, 'targets': '(15,)', 'init': 'ch52_bal_ft12_s1.pt', 'replay': '(12, 25, 35, 45)', 'replay_per_grid': 4000, 'best_epoch': 21, 'epochs_run': 31, 'minutes': 11.407, 'select_val_mse_db2': 1.4901, 'budget_limited': False, 'val15_mse_db2': 1.4901}


Training bal_ft35_s1:   0%|          | 0/60 [00:00<?, ?it/s]

Mounted at /content/drive
{'run_id': 'bal_ft35_s1', 'arm': 'bal_ft35', 'seed': 1, 'targets': '(35,)', 'init': 'ch52_bal_ft15_s1.pt', 'replay': '(12, 15, 25, 45)', 'replay_per_grid': 4000, 'best_epoch': 16, 'epochs_run': 26, 'minutes': 9.8975, 'select_val_mse_db2': 0.9647, 'budget_limited': False, 'val35_mse_db2': 0.9647}


Training bal_ft45_s1:   0%|          | 0/60 [00:00<?, ?it/s]

Mounted at /content/drive
{'run_id': 'bal_ft45_s1', 'arm': 'bal_ft45', 'seed': 1, 'targets': '(45,)', 'init': 'ch52_bal_ft35_s1.pt', 'replay': '(12, 15, 25, 35)', 'replay_per_grid': 4000, 'best_epoch': 9, 'epochs_run': 19, 'minutes': 7.3945, 'select_val_mse_db2': 0.8298, 'budget_limited': False, 'val45_mse_db2': 0.8298}
completed: 8 of 8


## 9. TEST TOUCH
The only cell that scores test graphs, guarded by a counter.

**What it scores:**
- every Chunk 52 checkpoint, on all five test sets
- `best_model.pt` zero-shot, on all five test sets

**Reproduction check:** `best_model` on 25×25 test must reproduce its recorded MSE 1.3175 and MAE 0.3742 (Chunk 36, `A_full`) to within 0.005.

**Writes:** `ch52_per_sample.parquet`, with errors, class and the **full predicted curve** for every new-grid test antenna.

In [10]:
# ── TEST TOUCH (the only cell that scores test graphs) ──
assert globals().get('HARNESS') is True and TEST_TOUCHES == 0; TEST_TOUCHES += 1
RUNS_DF = pd.read_csv(RUNS)
FINAL = rid('bal_ft45', 0)
assert FINAL in set(RUNS_DF.run_id), f'STOP: train {FINAL} first'
TRUE = {N: truth(S[N]['test']) for N in GRIDS}
PRED = {}
for rr in tqdm(list(RUNS_DF.itertuples()), desc='Scoring Checkpoints'):
    m = gnn(6); m.load_state_dict(state_of(rr.checkpoint), strict=True)
    for N in tqdm(GRIDS, desc=f'Evaluating {rr.run_id}', leave=False):
        PRED[(rr.run_id, N)] = predict(m, S[N]['test'])
    del m; torch.cuda.empty_cache()
for N in tqdm(GRIDS, desc='Evaluating Zero-Shot references'):
    PRED[('best_model', N)] = predict(BASE5, S[N]['test'], nf=5)
rows = []
for (mid, N), P in tqdm(PRED.items(), desc='Calculating per-sample metrics'):
    T = TRUE[N]
    for k, key in enumerate(S[N]['test']):
        rows.append(dict(model_id=mid, grid=N, source=key[0], sample_idx=key[1], cls=CLS.get(key, 'natural_25x25'),
                         true_min_db=T[k].min(), pred_min_db=P[k].min(), mse_db2=((P[k] - T[k]) ** 2).mean(), mae_db=np.abs(P[k] - T[k]).mean(),
                         y_hat=P[k].astype(np.float32).tolist() if N in NEW else None))
PER = pd.DataFrame(rows); PER.to_parquet(OUT('ch52_per_sample.parquet'), index=False)
print(f'TEST touched once: {len(PRED)} model-grid pairs; {len(PER):,} rows (predicted curves saved for the new grids)')
for k, ref in (('mse_db2', 1.3175), ('mae_db', 0.3742)):
    v = float(PER[(PER.model_id == 'best_model') & (PER.grid == 25)][k].mean())
    print(f'best_model 25x25 {k} {v:.4f} | Chunk 36 {ref:.4f}')
    assert abs(v - ref) < 0.005, 'STOP: best_model does not reproduce its recorded 25x25 test result'

Scoring Checkpoints:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating bal_ft12_s0:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating bal_ft15_s0:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating bal_ft35_s0:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating bal_ft45_s0:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating bal_ft12_s1:   0%|          | 0/5 [00:00<?, ?it/s]

... (2 outputs left out)

Evaluating bal_ft45_s1:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating Zero-Shot references:   0%|          | 0/5 [00:00<?, ?it/s]

Calculating per-sample metrics:   0%|          | 0/45 [00:00<?, ?it/s]

TEST touched once: 45 model-grid pairs; 107,847 rows (predicted curves saved for the new grids)
best_model 25x25 mse_db2 1.3175 | Chunk 36 1.3175
best_model 25x25 mae_db 0.3742 | Chunk 36 0.3742


## 10. Metrics, paired comparisons, statements
**Metrics per model and grid:** MSE, RMSE, MAE, R² and AUROC, plus MSE and MAE **per class**, with 95% stratified bootstrap intervals (1,000 resamples, rng 521).

**Paired comparisons** (same test antennas, rng 522):
- final − `best_model` zero-shot, on every grid (S1, and S2 at 25×25)
- each stage − the previous stage, on every grid, to show what each stage adds or costs

**Writes:** `ch52_metrics.csv`, `ch52_paired.csv`.

In [11]:
# ── Metrics with 95% intervals, per-class errors, paired comparisons, statements ──
R = 1000
def stats(P, T, rng):
    mse = ((P - T) ** 2).mean(1); mae = np.abs(P - T).mean(1); sse = ((P - T) ** 2).sum(1); y = (T.min(1) < TAU).astype(int)
    pt = dict(mse_db2=mse.mean(), rmse_db=np.sqrt(mse.mean()), mae_db=mae.mean(), r2=1 - sse.sum() / ((T - T.mean()) ** 2).sum(),
              auroc=roc_auc_score(y, -P.min(1)) if 0 < y.sum() < len(y) else np.nan)
    i0, i1 = np.flatnonzero(y == 0), np.flatnonzero(y == 1); reps = {k: np.empty(R) for k in ('mse_db2', 'mae_db', 'r2')}
    for r_ in range(R):
        ii = np.r_[rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]
        reps['mse_db2'][r_] = mse[ii].mean(); reps['mae_db'][r_] = mae[ii].mean()
        reps['r2'][r_] = 1 - sse[ii].sum() / ((T[ii] - T[ii].mean()) ** 2).sum()
    return pt, reps
rows = []
for (mid, N), P in PRED.items():
    pt, reps = stats(P, TRUE[N], np.random.default_rng(521))
    for k, v in pt.items():
        lo, hi = np.quantile(reps[k], [.025, .975]) if k in reps else (np.nan, np.nan)
        rows.append(dict(model_id=mid, grid=N, subset='all', metric=k, estimate=v, lo=lo, hi=hi))
    if N in NEW:
        d = PER[(PER.model_id == mid) & (PER.grid == N)]
        for c in ('single_band', 'non_functioning'):
            for k in ('mse_db2', 'mae_db'):
                x = d[d.cls == c][k].values; rng = np.random.default_rng(521)
                bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(R)]
                rows.append(dict(model_id=mid, grid=N, subset=c, metric=k, estimate=x.mean(), lo=np.quantile(bs, .025), hi=np.quantile(bs, .975)))
MET = pd.DataFrame(rows); MET.to_csv(OUT('ch52_metrics.csv'), index=False)
def get(mid, N, k, subset='all'): return MET[(MET.model_id == mid) & (MET.grid == N) & (MET.metric == k) & (MET.subset == subset)].iloc[0]
def per(mid, N): return PER[(PER.model_id == mid) & (PER.grid == N)].sort_values(['source', 'sample_idx'])
def paired(a, b, N):
    A, B = per(a, N), per(b, N); assert (A.sample_idx.values == B.sample_idx.values).all()
    y = A.true_min_db.values < TAU; i0, i1 = np.flatnonzero(~y), np.flatnonzero(y); out = []
    for k in ('mse_db2', 'mae_db'):
        d = A[k].values - B[k].values; rng = np.random.default_rng(522)
        reps = np.array([d[np.r_[rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]].mean() for _ in range(R)])
        lo, hi = np.quantile(reps, [.025, .975])
        out.append(dict(grid=N, A=a, B=b, metric=k, A_minus_B=d.mean(), lo=lo, hi=hi,
                        verdict='A better' if hi < 0 else ('B better' if lo > 0 else 'no significant difference')))
    return out
CHAIN = ['best_model', rid('bal_ft12', 0), rid('bal_ft15', 0), rid('bal_ft35', 0), FINAL]
prow = []
for N in GRIDS:
    prow += paired(FINAL, 'best_model', N)
    for a, b in zip(CHAIN[1:], CHAIN[:-1]):
        prow += paired(a, b, N)
PAIR = pd.DataFrame(prow); PAIR.to_csv(OUT('ch52_paired.csv'), index=False)
def v(a, b, N, k): return PAIR[(PAIR.A == a) & (PAIR.B == b) & (PAIR.grid == N) & (PAIR.metric == k)].iloc[0]
ANS = {}
ANS['S1 final vs zero-shot best_model (new test sets)'] = ' | '.join(
    f'{N}x{N}: MSE {get(FINAL, N, "mse_db2").estimate:.3f} vs {get("best_model", N, "mse_db2").estimate:.3f} ({v(FINAL, "best_model", N, "mse_db2").verdict}), '
    f'MAE {get(FINAL, N, "mae_db").estimate:.3f} vs {get("best_model", N, "mae_db").estimate:.3f} ({v(FINAL, "best_model", N, "mae_db").verdict})' for N in NEW)
x = v(FINAL, 'best_model', 25, 'mse_db2'); y_ = v(FINAL, 'best_model', 25, 'mae_db')
ANS['S2 25x25 cost of the chain (final - best_model)'] = (f'MSE {x.A_minus_B:+.4f} [{x.lo:+.4f}, {x.hi:+.4f}] ({x.verdict}); '
                                                         f'MAE {y_.A_minus_B:+.4f} [{y_.lo:+.4f}, {y_.hi:+.4f}] ({y_.verdict})')
ANS['S3 final model: single-band vs non-functioning MSE / MAE'] = ' | '.join(
    f'{N}x{N}: single {get(FINAL, N, "mse_db2", "single_band").estimate:.3f} / {get(FINAL, N, "mae_db", "single_band").estimate:.3f}, '
    f'non-func {get(FINAL, N, "mse_db2", "non_functioning").estimate:.3f} / {get(FINAL, N, "mae_db", "non_functioning").estimate:.3f}' for N in NEW)
print(PAIR.round(4).to_string(index=False)); print('\nPRE-REGISTERED STATEMENTS')
for k, t in ANS.items(): print(f'  {k}:\n      {t}')

 grid           A           B  metric  A_minus_B      lo      hi                   verdict
   12 bal_ft45_s0  best_model mse_db2    -3.1228 -3.4347 -2.7925                  A better
   12 bal_ft45_s0  best_model  mae_db    -0.3554 -0.3855 -0.3255                  A better
   12 bal_ft12_s0  best_model mse_db2    -3.1901 -3.5239 -2.8449                  A better
   12 bal_ft12_s0  best_model  mae_db    -0.3672 -0.4001 -0.3352                  A better
   12 bal_ft15_s0 bal_ft12_s0 mse_db2     0.0110 -0.0148  0.0346 no significant difference
   12 bal_ft15_s0 bal_ft12_s0  mae_db    -0.0000 -0.0042  0.0039 no significant difference
   12 bal_ft35_s0 bal_ft15_s0 mse_db2     0.0598  0.0304  0.0856                  B better
   12 bal_ft35_s0 bal_ft15_s0  mae_db     0.0082  0.0039  0.0119                  B better
   12 bal_ft45_s0 bal_ft35_s0 mse_db2    -0.0035 -0.0376  0.0322 no significant difference
   12 bal_ft45_s0 bal_ft35_s0  mae_db     0.0037 -0.0007  0.0084 no significant difference

## 11. Report
**Writes:** `ch52_report.md` and `ch52_table_values.csv`. The report holds:
- metrics with intervals for every model and grid, including the per-class errors
- the paired verdicts and S1–S3
- the run table
- the values the GATv2-vs-CNN table will need (final model, test MSE and MAE per grid)

In [12]:
# ── Report ──
def fmt(mid, N, k, subset='all'):
    r = get(mid, N, k, subset); return f'{r.estimate:.3f}' if np.isnan(r.lo) else f'{r.estimate:.3f} [{r.lo:.3f}, {r.hi:.3f}]'
TAB = pd.DataFrame([dict(grid=f'{N}x{N}', n_test=len(S[N]['test']), test_set='Chunk 51 balanced' if N in NEW else 'unchanged (natural)',
                         gatv2_mse=get(FINAL, N, 'mse_db2').estimate, gatv2_mae=get(FINAL, N, 'mae_db').estimate,
                         zero_shot_mse=get('best_model', N, 'mse_db2').estimate, zero_shot_mae=get('best_model', N, 'mae_db').estimate) for N in GRIDS])
TAB.to_csv(OUT('ch52_table_values.csv'), index=False)
L = ['# Chunk 52 — Replay-based fine-tuning on the balanced single-band datasets (Chunk 51)', '',
     f'Chain: best_model (+zero column) → bal_ft12 → bal_ft15 → bal_ft35 → `{FINAL}`; every stage replays the other four grids. One test touch; 95% stratified '
     'bootstrap intervals. New 12/15/35/45 test sets: 500 antennas each, ≈50% single-band / 50% non-functioning. 25×25 test unchanged.', '',
     '| grid | model | MSE (dB²) | MAE (dB) | R² | AUROC | MSE single-band | MSE non-functioning |', '|---|---|---|---|---|---|---|---|']
for N in GRIDS:
    for mid in CHAIN:
        if (mid, N) not in PRED: continue
        sb = fmt(mid, N, 'mse_db2', 'single_band') if N in NEW else '—'; nf = fmt(mid, N, 'mse_db2', 'non_functioning') if N in NEW else '—'
        L.append(f'| {N}×{N} | {mid} | {fmt(mid, N, "mse_db2")} | {fmt(mid, N, "mae_db")} | {fmt(mid, N, "r2")} | {fmt(mid, N, "auroc")} | {sb} | {nf} |')
L += ['', '## Paired comparisons (A − B, same test antennas)', PAIR.round(4).to_markdown(index=False), '',
      '## Pre-registered statements'] + [f'- **{k}:** {t}' for k, t in ANS.items()]
L += ['', '## Values for the GATv2-vs-CNN table', TAB.round(4).to_markdown(index=False), '',
      '_Results apply to single-band and non-functioning antennas only; not comparable with the natural-distribution table._',
      '', '## Runs', RUNS_DF.drop(columns=['checkpoint_sha256'], errors='ignore').round(4).to_markdown(index=False)]
open(OUT('ch52_report.md'), 'w').write('\n'.join(L) + '\n'); print('\n'.join(L))

# Chunk 52 — Replay-based fine-tuning on the balanced single-band datasets (Chunk 51)

Chain: best_model (+zero column) → bal_ft12 → bal_ft15 → bal_ft35 → `bal_ft45_s0`; every stage replays the other four grids. One test touch; 95% stratified bootstrap intervals. New 12/15/35/45 test sets: 500 antennas each, ≈50% single-band / 50% non-functioning. 25×25 test unchanged.

| grid | model | MSE (dB²) | MAE (dB) | R² | AUROC | MSE single-band | MSE non-functioning |
|---|---|---|---|---|---|---|---|
| 12×12 | best_model | 5.059 [4.686, 5.454] | 0.850 [0.815, 0.886] | 0.064 [-0.026, 0.148] | 0.741 | 6.653 [6.053, 7.344] | 3.465 [3.082, 3.921] |
| 12×12 | bal_ft12_s0 | 1.869 [1.696, 2.059] | 0.483 [0.463, 0.505] | 0.654 [0.620, 0.686] | 0.747 | 3.001 [2.652, 3.393] | 0.737 [0.658, 0.822] |
| 12×12 | bal_ft15_s0 | 1.880 [1.706, 2.066] | 0.483 [0.463, 0.505] | 0.652 [0.616, 0.685] | 0.746 | 2.997 [2.651, 3.371] | 0.762 [0.680, 0.851] |
| 12×12 | bal_ft35_s0 | 1.939 [1.770, 2.121] | 0.491 [0.472

## 12. Guards
**What it checks:**
- one test touch
- every Chunk 52 checkpoint matches its recorded SHA-256
- the chain order: base → A (12) → B (15) → C (35) → D (45)
- no run was stopped by its epoch cap
- predicted curves were saved for every new-grid test antenna
- input files unchanged, and no earlier file modified
- every output named `ch52_*`

In [13]:
# ── Guards ──
fails = []
def guard(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok: fails.append(msg)
R2 = pd.read_csv(RUNS)
guard(TEST_TOUCHES == 1, 'TEST touched exactly once')
guard(all(sha(p) == h for p, h in zip(R2.checkpoint, R2.checkpoint_sha256)), 'every ch52 checkpoint matches its recorded SHA-256')
exp_init = {'bal_ft12': 'best_model.pt (+zero column)', 'bal_ft15': 'ch52_bal_ft12_s{s}.pt', 'bal_ft35': 'ch52_bal_ft15_s{s}.pt', 'bal_ft45': 'ch52_bal_ft35_s{s}.pt'}
guard(all(r.init == exp_init[r.arm].format(s=r.seed) for r in R2.itertuples()), 'chain: best_model -> A (12) -> B (15) -> C (35) -> D (45)')
guard(not R2.budget_limited.any(), 'no run stopped by its epoch cap' if not R2.budget_limited.any() else
      f'BUDGET-LIMITED runs: {list(R2[R2.budget_limited].run_id)} (best epoch within 10 of the 60-epoch cap)')
P2 = pd.read_parquet(OUT('ch52_per_sample.parquet'))
guard(P2[P2.grid.isin(NEW)].y_hat.map(lambda a: a is not None and len(a) == 201).all(), 'predicted curves saved for every new-grid test antenna')
guard(all(sha(p) == h for p, h in READ_SHA.items() if not Path(p).name.startswith('ch52_')), f'input files unchanged ({len(READ_SHA)})')
guard(all(os.path.getmtime(p) == t for p, t in PRIOR.items() if os.path.exists(p)), f'no earlier file modified ({len(PRIOR)})')
guard(all(Path(p).name.startswith('ch52_') for p in glob.glob(f'{ART}/ch52_*') + glob.glob(f'{CKPT}/ch52_*')), 'all outputs ch52_*')
assert not fails, fails
print('ALL GUARDS PASSED')

  [PASS] TEST touched exactly once
  [PASS] every ch52 checkpoint matches its recorded SHA-256
  [PASS] chain: best_model -> A (12) -> B (15) -> C (35) -> D (45)
  [PASS] no run stopped by its epoch cap
  [PASS] predicted curves saved for every new-grid test antenna
  [PASS] input files unchanged (31)
  [PASS] no earlier file modified (298)
  [PASS] all outputs ch52_*
ALL GUARDS PASSED
